# SVG LoRA — Inference + Ablation Study (Colab)

**No training required.** Loads saved model weights directly.

## What you need to upload to Colab:
1. `fastmodel.zip` → upload and unzip to `/content/fastmodel/`
2. `test.csv` → upload to `/content/test.csv`

## Sections:
1. Setup & upload instructions
2. Load model from saved weights
3. Generate submission (1000 prompts)
4. Ablation 1 — Token Budget: 220 vs 500
5. Ablation 2 — Base Model vs Fine-tuned
6. Ablation 3 — Repetition Penalty: 1.0 vs 1.1
7. Ablation summary

---
## Step 1: Install dependencies

In [3]:
!pip install -q transformers peft accelerate bitsandbytes datasets trl safetensors pandas lxml

## Step 2: Upload files

Run this cell to upload `fastmodel.zip` and `test.csv`.

In [4]:
import zipfile, os

# unzip
with zipfile.ZipFile('/content/fastmodel.zip', 'r') as z:
    z.extractall('/content/')
print('Unzipped fastmodel.zip')

# quick file check
print('\nFiles in /content/fastmodel/:')
for f in os.listdir('/content/fastmodel/'):
    print(' ', f)

print('\ntest.csv exists:', os.path.exists('/content/test.csv'))

Unzipped fastmodel.zip

Files in /content/fastmodel/:
  adapter_config.json
  README.md
  adapter_model.safetensors
  .DS_Store
  improved-svg-lora-v2-3.ipynb
  tokenizer.json
  chat_template.jinja
  training_args
  tokenizer_config.json

test.csv exists: True


## Step 3: Imports and config

In [5]:
import os, re, csv, random
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

print('Torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# paths — edit for your env
BASE_MODEL   = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'
ADAPTER_PATH = '/content/fastmodel'       # unzipped adapter root
TEST_PATH    = '/content/test.csv'
OUTPUT_DIR   = '/content/'

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

print('\nPaths set.')
print('ADAPTER_PATH exists:', os.path.exists(ADAPTER_PATH))
print('TEST_PATH exists    :', os.path.exists(TEST_PATH))

Torch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4

Paths set.
ADAPTER_PATH exists: True
TEST_PATH exists    : True


## Step 4: Helper functions

In [6]:
ALLOWED_TAGS = {
    'svg','g','path','rect','circle','ellipse','line','polyline','polygon',
    'defs','use','symbol','clipPath','mask','linearGradient','radialGradient',
    'stop','text','tspan','title','desc','style','pattern','marker','filter',
}
DISALLOWED_ATTRS_RE = re.compile(
    r'\s+(?:filling|onclick|onload|onmouseover|xmlns:xlink)=["\'][^"\']*("|\')',
    re.IGNORECASE
)
SVG_REGEX = re.compile(r'(<svg\b[\s\S]*?</svg>)', flags=re.IGNORECASE)

def normalize_svg(svg_text):
    if not svg_text: return ''
    svg_text = DISALLOWED_ATTRS_RE.sub('', svg_text)
    for attr in ('width', 'height', 'viewBox'):
        svg_text = re.sub(rf'(<svg\b[^>]*?)\b{attr}=["\'][^"\']*("|\')', r'\1', svg_text)
    svg_text = re.sub(
        r'(<svg\b)',
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"',
        svg_text, count=1
    )
    svg_text = re.sub(r'(xmlns="[^"]*")(?=.*xmlns="[^"]*")', '', svg_text)
    return re.sub(r'\s+', ' ', svg_text).strip()

def has_disallowed_tags(svg_text):
    try:
        root = ET.fromstring(svg_text)
        for elem in root.iter():
            local = elem.tag.split('}')[-1] if '}' in elem.tag else elem.tag
            if local not in ALLOWED_TAGS: return True
        return False
    except Exception: return True

def is_valid_svg(svg_text):
    if not svg_text: return False
    try:
        root = ET.fromstring(svg_text)
        tag = root.tag.split('}')[-1] if '}' in root.tag else root.tag
        return tag == 'svg'
    except ET.ParseError: return False

def extract_svg(text):
    if not text: return ''
    for tok in ['<|im_end|>', '<|endoftext|>']:
        idx = text.find(tok)
        if idx != -1: text = text[:idx]
    match = SVG_REGEX.search(text)
    if match: return match.group(1).strip()
    start = text.lower().find('<svg')
    if start != -1:
        partial = text[start:].strip()
        if not partial.lower().endswith('</svg>'): partial += '</svg>'
        return partial
    return ''

def fallback_svg():
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        '<rect x="0" y="0" width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="64" fill="#cccccc"/>'
        '</svg>'
    )

SYSTEM_PROMPT = (
    'You are an SVG code generator. '
    'Given a text description, output ONLY valid SVG code with '
    'width="256" height="256" viewBox="0 0 256 256". '
    'Do not include explanations or markdown code fences.'
)

print('Helper functions defined.')

Helper functions defined.


## Step 5: Load fine-tuned model from saved weights

In [7]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

ft_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
ft_model = PeftModel.from_pretrained(ft_model, ADAPTER_PATH, is_trainable=False)
ft_model.eval()
ft_model.config.use_cache = True
if hasattr(ft_model, 'gradient_checkpointing_disable'):
    ft_model.gradient_checkpointing_disable()

print('Fine-tuned model loaded from:', ADAPTER_PATH)
print('Device:', next(ft_model.parameters()).device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Fine-tuned model loaded from: /content/fastmodel
Device: cuda:0


## Step 6: Load test data

In [8]:
test_df = pd.read_csv(TEST_PATH, engine='python', quoting=csv.QUOTE_MINIMAL)
print(f'Test set: {len(test_df)} rows')
print(f'Columns: {test_df.columns.tolist()}')
test_df.head(3)

Test set: 1000 rows
Columns: ['id', 'prompt']


,id,prompt
0,fa1d8fa7-080f-4269-a9cf-a17562c9a0ca,firewood stack cut logs wood with leaf illustr...
1,6eede943219547c22ac56085027d33cc,The image shows five horizontal lines of varyi...
2,ea045c7a247166f061ce504d9b7ccaab,A stylized icon depicting a curved arrow withi...


---
# Ablation Study

All ablations run on the **first 50 test prompts** for speed (~3-5 min each).

- **Ablation 1**: Token Budget — `max_new_tokens=220` vs `500`
- **Ablation 2**: Base Model (no LoRA) vs Fine-tuned
- **Ablation 3**: Repetition Penalty — `1.0` vs `1.1`

In [14]:
# ablation: first 20 prompts only
ABL_PROMPTS = test_df['prompt'].iloc[:20].tolist()
print(f'Ablation subset: {len(ABL_PROMPTS)} prompts')

def run_ablation(model_to_use, prompts, max_new_tokens=500,
                 repetition_penalty=1.1, label=''):
    """Run prompts, return dict with valid_pct, fallback_pct, avg_len."""
    svgs, fallbacks = [], []

    for i, prompt in enumerate(prompts):
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': prompt},
        ]
        chat_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(
            chat_text, return_tensors='pt',
            truncation=True, max_length=512
        ).to(model_to_use.device)
        input_len = inputs['input_ids'].shape[1]

        with torch.no_grad():
            with torch.amp.autocast('cuda'):
                output_ids = model_to_use.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    repetition_penalty=repetition_penalty,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                    use_cache=True,
                )

        decoded = tokenizer.decode(
            output_ids[0][input_len:], skip_special_tokens=False
        )
        svg = normalize_svg(extract_svg(decoded))
        bad = not is_valid_svg(svg) or has_disallowed_tags(svg) or len(svg) > 15900
        if bad: svg = fallback_svg()
        svgs.append(svg)
        fallbacks.append(bad)

        if (i + 1) % 10 == 0:
            print(f'  [{label}] {i+1}/{len(prompts)} done')

    n            = len(prompts)
    valid_pct    = sum(is_valid_svg(s) for s in svgs) / n * 100
    fallback_pct = sum(fallbacks) / n * 100
    avg_len      = sum(len(s) for s in svgs) / n
    print(f'  -> [{label}] valid={valid_pct:.1f}%  fallback={fallback_pct:.1f}%  avg_len={avg_len:.0f}')
    return {'label': label, 'valid_pct': valid_pct,
            'fallback_pct': fallback_pct, 'avg_len': avg_len}

print('run_ablation() ready.')

Ablation subset: 20 prompts
run_ablation() ready.


### Ablation 1: Token Budget (220 vs 500)

In [15]:
print('=== Ablation 1: Token Budget ===')

print('\n[1/2] Running tokens=220 ...')
r_220 = run_ablation(ft_model, ABL_PROMPTS, max_new_tokens=220, label='tokens=220')

print('\n[2/2] Running tokens=500 ...')
r_500 = run_ablation(ft_model, ABL_PROMPTS, max_new_tokens=500, label='tokens=500')

print('\nResult:')
print(pd.DataFrame([r_220, r_500])[['label','valid_pct','fallback_pct','avg_len']].to_string(index=False))

=== Ablation 1: Token Budget ===

[1/2] Running tokens=220 ...
  [tokens=220] 10/20 done
  [tokens=220] 20/20 done
  -> [tokens=220] valid=100.0%  fallback=80.0%  avg_len=204

[2/2] Running tokens=500 ...
  [tokens=500] 10/20 done
  [tokens=500] 20/20 done
  -> [tokens=500] valid=100.0%  fallback=80.0%  avg_len=204

Result:
     label  valid_pct  fallback_pct  avg_len
tokens=220      100.0          80.0   203.65
tokens=500      100.0          80.0   203.65


### Ablation 2: Base Model (no LoRA) vs Fine-tuned

In [16]:
print('=== Ablation 2: Base Model vs Fine-tuned ===')

# base weights only (compare to LoRA)
print('\n[1/2] Loading base model (no adapter)...')
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
base_model.eval()
base_model.config.use_cache = True
print('Base model loaded. Running inference...')

r_base = run_ablation(base_model, ABL_PROMPTS, max_new_tokens=500, label='base (no LoRA)')

# free VRAM
del base_model
torch.cuda.empty_cache()
print('Base model deleted, GPU memory freed.')

print('\n[2/2] Running fine-tuned model (v3)...')
r_ft = run_ablation(ft_model, ABL_PROMPTS, max_new_tokens=500, label='fine-tuned (v3)')

print('\nResult:')
print(pd.DataFrame([r_base, r_ft])[['label','valid_pct','fallback_pct','avg_len']].to_string(index=False))

=== Ablation 2: Base Model vs Fine-tuned ===

[1/2] Loading base model (no adapter)...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Base model loaded. Running inference...
  [base (no LoRA)] 10/20 done
  [base (no LoRA)] 20/20 done
  -> [base (no LoRA)] valid=100.0%  fallback=30.0%  avg_len=288
Base model deleted, GPU memory freed.

[2/2] Running fine-tuned model (v3)...
  [fine-tuned (v3)] 10/20 done
  [fine-tuned (v3)] 20/20 done
  -> [fine-tuned (v3)] valid=100.0%  fallback=80.0%  avg_len=204

Result:
          label  valid_pct  fallback_pct  avg_len
 base (no LoRA)      100.0          30.0   288.10
fine-tuned (v3)      100.0          80.0   203.65


### Ablation 3: Repetition Penalty (1.0 vs 1.1)

In [17]:
print('=== Ablation 3: Repetition Penalty ===')

print('\n[1/2] Running rep_penalty=1.0 ...')
r_no_rep = run_ablation(ft_model, ABL_PROMPTS, max_new_tokens=500,
                         repetition_penalty=1.0, label='rep_penalty=1.0')

print('\n[2/2] Running rep_penalty=1.1 ...')
r_rep    = run_ablation(ft_model, ABL_PROMPTS, max_new_tokens=500,
                         repetition_penalty=1.1, label='rep_penalty=1.1')

print('\nResult:')
print(pd.DataFrame([r_no_rep, r_rep])[['label','valid_pct','fallback_pct','avg_len']].to_string(index=False))

=== Ablation 3: Repetition Penalty ===

[1/2] Running rep_penalty=1.0 ...
  [rep_penalty=1.0] 10/20 done
  [rep_penalty=1.0] 20/20 done
  -> [rep_penalty=1.0] valid=100.0%  fallback=90.0%  avg_len=208

[2/2] Running rep_penalty=1.1 ...
  [rep_penalty=1.1] 10/20 done
  [rep_penalty=1.1] 20/20 done
  -> [rep_penalty=1.1] valid=100.0%  fallback=80.0%  avg_len=204

Result:
          label  valid_pct  fallback_pct  avg_len
rep_penalty=1.0      100.0          90.0   208.20
rep_penalty=1.1      100.0          80.0   203.65


### Full Ablation Summary

In [20]:
all_results = [r_220, r_500, r_base, r_ft, r_no_rep, r_rep]

summary = pd.DataFrame([
    {
        'Ablation': (
            'Token Budget'       if 'tokens' in r['label'] else
            'Base vs Fine-tuned' if 'base'   in r['label'] or 'fine' in r['label'] else
            'Repetition Penalty'
        ),
        'Config':        r['label'],
        'Valid SVG (%)': f"{r['valid_pct']:.1f}",
        'Fallback (%)':  f"{r['fallback_pct']:.1f}",
        'Avg SVG len':   f"{r['avg_len']:.0f}",
    }
    for r in all_results
])

print('==================== ABLATION SUMMARY ====================')
print(summary.to_string(index=False))

# write CSV
summary.to_csv('/content/ablation_results.csv', index=False)
print('\nSaved: /content/ablation_results.csv')

# optional download
from google.colab import files
files.download('/content/ablation_results.csv')

==================== ABLATION SUMMARY ====================
          Ablation          Config Valid SVG (%) Fallback (%) Avg SVG len
      Token Budget      tokens=220         100.0         80.0         204
      Token Budget      tokens=500         100.0         80.0         204
Base vs Fine-tuned  base (no LoRA)         100.0         30.0         288
Base vs Fine-tuned fine-tuned (v3)         100.0         80.0         204
Repetition Penalty rep_penalty=1.0         100.0         90.0         208
Repetition Penalty rep_penalty=1.1         100.0         80.0         204

Saved: /content/ablation_results.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>